In [5]:
import pandas as pd # type:ignore
import numpy as np # type:ignore
import matplotlib.pyplot as plt # type:ignore
import seaborn as sns # type:ignore

In [16]:
df_hr = pd.read_csv('rogii-wellbore-geology-prediction/train/0a57a29c__horizontal_well.csv')
df_vr = pd.read_csv('rogii-wellbore-geology-prediction/train/0a57a29c__typewell.csv')

In [17]:
df_hr[5000:5005]

,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input
5000,15712.0,2900791.23,1034355.66,-9040.14,-8593.18,-8816.62,-8883.00,-8932.52,-8970.55,-9096.87,11209.46,51.876231,NaN
5001,15713.0,2900790.58,1034356.42,-9040.11,-8593.15,-8816.59,-8882.97,-8932.49,-8970.52,-9096.84,11209.46,48.183973,NaN
5002,15714.0,2900789.93,1034357.18,-9040.09,-8593.12,-8816.56,-8882.94,-8932.46,-8970.50,-9096.81,11209.46,NaN,NaN
5003,15715.0,2900789.28,1034357.94,-9040.06,-8593.09,-8816.53,-8882.91,-8932.43,-8970.47,-9096.78,11209.47,NaN,NaN
5004,15716.0,2900788.63,1034358.70,-9040.04,-8593.07,-8816.51,-8882.89,-8932.40,-8970.44,-9096.76,11209.47,NaN,NaN


In [18]:
df_vr.head()

,TVT,GR,Geology
0,10669.45,94.57,NaN
1,10669.95,87.15,NaN
2,10670.45,84.14,NaN
3,10670.95,82.09,NaN
4,10671.45,83.81,NaN


In [19]:
print(df_hr.shape)
print(df_vr.shape)

(7194, 13)
(1395, 3)


In [20]:
vr_sorted = df_vr.sort_values(by='TVT').reset_index(drop=True)

In [21]:
vr_lookup = vr_sorted[['TVT', 'GR']].rename(columns={'GR': 'expected_vertical_GR'})

In [22]:
# 1. Sort the horizontal file by depth 'Z' (required for merge_asof)
hr_sorted = df_hr.sort_values(by='Z')

# 2. Match them up using the closest vertical depth
combined_df = pd.merge_asof(
    hr_sorted, 
    vr_lookup,
    left_on='Z',          # Look at our current depth
    right_on='TVT',        # Find the closest depth in the master template
    direction='nearest'   # Grab the closest number
)

# 3. Put the horizontal well back in its original sequence order
combined_df = combined_df.sort_values(by='MD').reset_index(drop=True)